# MRI classification: grouped component ablations

Seven prespecified variants, five frozen grouped outer folds and three training seeds.
The data have no verified patient identifiers: these are **image-level exploratory results**.
Read `EXPERIMENT_PLAN.md`, `docs/NOTEBOOK_AUDIT.md` and the dataset documentation before reporting.
The source snapshot is included below so Kaggle runs exactly the reviewed Python code.


## Exact source snapshot
The separate source files are the review/edit interface; this generated cell makes the notebook portable.

In [ ]:
# This cell is generated from the readable Python source in the GitHub package.
import hashlib, json, os, sys
from pathlib import Path
WORK = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path.cwd()
SOURCE_ROOT = WORK / 'mri_source'
SOURCE_ROOT.mkdir(parents=True, exist_ok=True)
SOURCES = {'classification/__init__.py': '', 'classification/models.py': '"""Models adapted from the supplied classification notebook; see audit for provenance.\nThe no-multiscale control keeps all branches and parameters but uses dilation 1.\nResNet2D18 is the original narrow GroupNorm implementation, not torchvision ResNet.\n"""\nimport torch\nfrom torch import nn\n\nclass CONFIG:\n    IN_CHANNELS=1\n    BASE_CHANNELS=16\n    DROPOUT=0.4\n\ndef norm_layer(channels):\n    for g in (8, 4, 2, 1):\n        if channels % g == 0:\n            return nn.GroupNorm(g, channels)\n    return nn.GroupNorm(1, channels)\n\ndef act_layer():\n    return nn.ReLU(inplace=False)\n\nclass ChannelAttention2D(nn.Module):\n\n    def __init__(self, channels, reduction=8):\n        super().__init__()\n        self.avg_pool = nn.AdaptiveAvgPool2d(1)\n        self.max_pool = nn.AdaptiveMaxPool2d(1)\n        hidden = max(channels // reduction, 4)\n        self.mlp = nn.Sequential(nn.Conv2d(channels, hidden, 1, bias=False), act_layer(), nn.Conv2d(hidden, channels, 1, bias=False))\n        self.sigmoid = nn.Sigmoid()\n\n    def forward(self, x):\n        return self.sigmoid(self.mlp(self.avg_pool(x)) + self.mlp(self.max_pool(x)))\n\nclass SpatialAttention2D(nn.Module):\n\n    def __init__(self, kernel_size=7):\n        super().__init__()\n        self.conv = nn.Conv2d(2, 1, kernel_size, padding=kernel_size // 2, bias=False)\n        self.sigmoid = nn.Sigmoid()\n\n    def forward(self, x):\n        avg_out = torch.mean(x, dim=1, keepdim=True)\n        max_out, _ = torch.max(x, dim=1, keepdim=True)\n        return self.sigmoid(self.conv(torch.cat([avg_out, max_out], dim=1)))\n\nclass CBAM2D(nn.Module):\n\n    def __init__(self, channels, reduction=8, kernel_size=7):\n        super().__init__()\n        self.ca = ChannelAttention2D(channels, reduction)\n        self.sa = SpatialAttention2D(kernel_size)\n\n    def forward(self, x):\n        x = x * self.ca(x)\n        x = x * self.sa(x)\n        return x\n\nclass MultiScaleResBlock2D(nn.Module):\n\n    def __init__(self, in_ch, out_ch, stride=1, use_attention=True, multiscale=True):\n        super().__init__()\n        if out_ch < 3:\n            raise ValueError(\'out_ch must be at least 3 for the three-branch block\')\n        branch_ch = out_ch // 3\n        rem = out_ch - branch_ch * 3\n        self.b1 = nn.Conv2d(in_ch, branch_ch, 3, stride=stride, padding=1, dilation=1, bias=False)\n        self.b2 = nn.Conv2d(in_ch, branch_ch, 3, stride=stride, padding=2 if multiscale else 1, dilation=2 if multiscale else 1, bias=False)\n        self.b3 = nn.Conv2d(in_ch, branch_ch + rem, 3, stride=stride, padding=3 if multiscale else 1, dilation=3 if multiscale else 1, bias=False)\n        self.bn1 = norm_layer(out_ch)\n        self.act = act_layer()\n        self.fuse = nn.Conv2d(out_ch, out_ch, 1, bias=False)\n        self.bn2 = norm_layer(out_ch)\n        self.use_attention = use_attention\n        self.attn = CBAM2D(out_ch) if use_attention else None\n        self.shortcut = None\n        if stride != 1 or in_ch != out_ch:\n            self.shortcut = nn.Sequential(nn.Conv2d(in_ch, out_ch, 1, stride=stride, bias=False), norm_layer(out_ch))\n\n    def forward(self, x):\n        identity = x\n        out = torch.cat([self.b1(x), self.b2(x), self.b3(x)], dim=1)\n        out = self.act(self.bn1(out))\n        out = self.bn2(self.fuse(out))\n        if self.attn is not None:\n            out = self.attn(out)\n        if self.shortcut is not None:\n            identity = self.shortcut(identity)\n        return self.act(out + identity)\n\ndef init_weights(module):\n    for m in module.modules():\n        if isinstance(m, nn.Conv2d):\n            nn.init.kaiming_normal_(m.weight, mode=\'fan_out\', nonlinearity=\'relu\')\n            if m.bias is not None:\n                nn.init.zeros_(m.bias)\n        elif isinstance(m, nn.GroupNorm):\n            nn.init.ones_(m.weight)\n            nn.init.zeros_(m.bias)\n        elif isinstance(m, nn.Linear):\n            nn.init.xavier_uniform_(m.weight)\n            nn.init.zeros_(m.bias)\n\nclass MSCANet2D(nn.Module):\n\n    def __init__(self, num_classes, in_channels=None, base=None, use_attention=True, dropout=None, multiscale=True):\n        super().__init__()\n        in_channels = CONFIG.IN_CHANNELS if in_channels is None else in_channels\n        base = CONFIG.BASE_CHANNELS if base is None else base\n        dropout = CONFIG.DROPOUT if dropout is None else dropout\n        self.stem = nn.Sequential(nn.Conv2d(in_channels, base, 3, stride=1, padding=1, bias=False), norm_layer(base), act_layer(), nn.MaxPool2d(2))\n        chs = [base, base * 2, base * 4, base * 8]\n        self.stage1 = MultiScaleResBlock2D(chs[0], chs[1], stride=2, use_attention=use_attention, multiscale=multiscale)\n        self.stage2 = MultiScaleResBlock2D(chs[1], chs[2], stride=2, use_attention=use_attention, multiscale=multiscale)\n        self.stage3 = MultiScaleResBlock2D(chs[2], chs[3], stride=2, use_attention=use_attention, multiscale=multiscale)\n        self.stage4 = MultiScaleResBlock2D(chs[3], chs[3], stride=2, use_attention=use_attention, multiscale=multiscale)\n        self.gap = nn.AdaptiveAvgPool2d(1)\n        self.classifier = nn.Sequential(nn.Flatten(), nn.Dropout(dropout), nn.Linear(chs[3], 128), act_layer(), nn.Dropout(dropout / 2), nn.Linear(128, num_classes))\n        self.target_layer_name = \'stage4\'\n        init_weights(self)\n\n    def forward(self, x):\n        x = self.stem(x)\n        x = self.stage1(x)\n        x = self.stage2(x)\n        x = self.stage3(x)\n        x = self.stage4(x)\n        return self.classifier(self.gap(x))\n\nclass Plain2DCNN(nn.Module):\n\n    def __init__(self, num_classes, in_channels=None, base=None, dropout=None):\n        super().__init__()\n        in_channels = CONFIG.IN_CHANNELS if in_channels is None else in_channels\n        base = CONFIG.BASE_CHANNELS if base is None else base\n        dropout = CONFIG.DROPOUT if dropout is None else dropout\n\n        def block(cin, cout):\n            return nn.Sequential(nn.Conv2d(cin, cout, 3, padding=1, bias=False), norm_layer(cout), act_layer(), nn.MaxPool2d(2))\n        self.features = nn.Sequential(block(in_channels, base), block(base, base * 2), block(base * 2, base * 4), block(base * 4, base * 8), block(base * 8, base * 8))\n        self.gap = nn.AdaptiveAvgPool2d(1)\n        self.classifier = nn.Sequential(nn.Flatten(), nn.Dropout(dropout), nn.Linear(base * 8, 128), act_layer(), nn.Linear(128, num_classes))\n        self.target_layer_name = \'features\'\n        init_weights(self)\n\n    def forward(self, x):\n        return self.classifier(self.gap(self.features(x)))\n\nclass BasicBlock2D(nn.Module):\n    expansion = 1\n\n    def __init__(self, in_ch, out_ch, stride=1):\n        super().__init__()\n        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, stride=stride, padding=1, bias=False)\n        self.bn1 = norm_layer(out_ch)\n        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False)\n        self.bn2 = norm_layer(out_ch)\n        self.act = act_layer()\n        self.shortcut = None\n        if stride != 1 or in_ch != out_ch:\n            self.shortcut = nn.Sequential(nn.Conv2d(in_ch, out_ch, 1, stride=stride, bias=False), norm_layer(out_ch))\n\n    def forward(self, x):\n        identity = x\n        out = self.act(self.bn1(self.conv1(x)))\n        out = self.bn2(self.conv2(out))\n        if self.shortcut is not None:\n            identity = self.shortcut(identity)\n        return self.act(out + identity)\n\nclass ResNet2D18(nn.Module):\n\n    def __init__(self, num_classes, in_channels=None, base=None):\n        super().__init__()\n        in_channels = CONFIG.IN_CHANNELS if in_channels is None else in_channels\n        base = CONFIG.BASE_CHANNELS if base is None else base\n        self.stem = nn.Sequential(nn.Conv2d(in_channels, base, 7, stride=2, padding=3, bias=False), norm_layer(base), act_layer(), nn.MaxPool2d(3, stride=2, padding=1))\n        self.layer1 = nn.Sequential(BasicBlock2D(base, base), BasicBlock2D(base, base))\n        self.layer2 = nn.Sequential(BasicBlock2D(base, base * 2, stride=2), BasicBlock2D(base * 2, base * 2))\n        self.layer3 = nn.Sequential(BasicBlock2D(base * 2, base * 4, stride=2), BasicBlock2D(base * 4, base * 4))\n        self.layer4 = nn.Sequential(BasicBlock2D(base * 4, base * 8, stride=2), BasicBlock2D(base * 8, base * 8))\n        self.gap = nn.AdaptiveAvgPool2d(1)\n        self.classifier = nn.Sequential(nn.Flatten(), nn.Linear(base * 8, num_classes))\n        self.target_layer_name = \'layer4\'\n        init_weights(self)\n\n    def forward(self, x):\n        x = self.stem(x)\n        x = self.layer1(x)\n        x = self.layer2(x)\n        x = self.layer3(x)\n        x = self.layer4(x)\n        return self.classifier(self.gap(x))\n\ndef build_model(name, num_classes):\n    if name == \'MSCANet2D\':\n        return MSCANet2D(num_classes, use_attention=True)\n    if name == \'MSCANet2D_NoAttention\':\n        return MSCANet2D(num_classes, use_attention=False)\n    if name == \'Plain2DCNN\':\n        return Plain2DCNN(num_classes)\n    if name == \'ResNet2D18\':\n        return ResNet2D18(num_classes)\n    raise ValueError(f\'Unknown model name: {name}\')\n', 'classification/study.py': '"""Deduplicated, grouped, repeated-seed classification benchmark."""\nfrom __future__ import annotations\n\nimport argparse\nimport hashlib\nimport json\nimport math\nimport os\nimport platform\nimport random\nimport time\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nfrom PIL import Image, ImageOps\nfrom scipy.fft import dctn\nfrom sklearn.metrics import (accuracy_score, average_precision_score,\n    balanced_accuracy_score, brier_score_loss, confusion_matrix, f1_score,\n    log_loss, roc_auc_score)\nfrom sklearn.model_selection import StratifiedGroupKFold\nimport torch\nfrom torch import nn\nfrom torch.nn import functional as F\n\nfrom classification.models import MSCANet2D, Plain2DCNN, ResNet2D18\n\nVARIANTS = ["full", "no_attention", "no_multiscale", "no_attention_no_multiscale",\n            "no_augmentation", "plain_cnn", "narrow_resnet_gn"]\n\n\ndef write_json(path, data):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    path.write_text(json.dumps(data, indent=2, sort_keys=True, allow_nan=False))\n\n\ndef sha(data):\n    return hashlib.sha256(data).hexdigest()\n\n\ndef discover_root(base):\n    base = Path(base)\n    if any(p.parent.name.lower() in {"yes", "no"} for p in base.rglob("*.jpg")):\n        return base\n    raise FileNotFoundError(f"No yes/no image dataset under {base}")\n\n\ndef audit_images(root, out):\n    """Identify bytes/pixels exactly; conservatively group approximate copies."""\n    root, out = Path(root), Path(out)\n    out.mkdir(parents=True, exist_ok=True)\n    rows, invalid = [], []\n    for p in sorted(root.rglob("*")):\n        if p.suffix.lower() not in {".png", ".jpg", ".jpeg"} or p.parent.name.lower() not in {"yes", "no"}:\n            continue\n        try:\n            with Image.open(p) as im:\n                im = ImageOps.exif_transpose(im).convert("L")\n                a = np.asarray(im)\n                tiny = np.asarray(im.resize((32, 32), Image.Resampling.BILINEAR), dtype=float)\n                d = dctn(tiny, norm="ortho")[:8, :8].ravel()[1:]\n                ph = sum(int(v) << i for i, v in enumerate(d > np.median(d)))\n                rows.append({"relative_path": p.relative_to(root).as_posix(),\n                    "label": p.parent.name.lower(), "y": int(p.parent.name.lower() == "yes"),\n                    "byte_hash": sha(p.read_bytes()),\n                    "image_id": sha(str(a.shape).encode() + a.tobytes()),\n                    "width": im.width, "height": im.height, "phash": ph, "tiny": tiny.ravel()})\n        except (OSError, ValueError) as exc:\n            invalid.append({"relative_path": p.relative_to(root).as_posix(), "error": str(exc)})\n    if not rows:\n        raise ValueError("No readable labeled images")\n    raw = pd.DataFrame(rows)\n    conflicts = raw.groupby("image_id").y.nunique()\n    bad = set(conflicts[conflicts > 1].index)\n    clean = raw[~raw.image_id.isin(bad)].drop_duplicates("image_id").reset_index(drop=True)\n    parent = list(range(len(clean)))\n\n    def find(a):\n        while parent[a] != a:\n            parent[a] = parent[parent[a]]\n            a = parent[a]\n        return a\n\n    candidates = []\n    for i in range(len(clean)):\n        for j in range(i):\n            distance = (int(clean.iloc[i].phash) ^ int(clean.iloc[j].phash)).bit_count()\n            if distance > 8:\n                continue\n            a, b = clean.iloc[i].tiny, clean.iloc[j].tiny\n            corr = float(np.corrcoef(a, b)[0, 1]) if min(a.std(), b.std()) > 0 else 0.0\n            grouped = distance <= 4 and corr >= 0.98\n            candidates.append({"image_a": clean.iloc[j].image_id, "image_b": clean.iloc[i].image_id,\n                               "phash_distance": distance, "correlation": corr, "grouped": grouped})\n            if grouped:\n                parent[find(i)] = find(j)\n    components = {}\n    for i in range(len(clean)):\n        components.setdefault(find(i), []).append(clean.iloc[i].image_id)\n    clean["group_id"] = [min(components[find(i)]) for i in range(len(clean))]\n    raw.drop(columns=["tiny"]).to_csv(out / "all_files.csv", index=False)\n    clean = clean.drop(columns=["tiny"])\n    clean.to_csv(out / "image_manifest.csv", index=False)\n    pd.DataFrame(candidates).to_csv(out / "similarity_candidates.csv", index=False)\n    report = {"raw_files": len(raw), "byte_unique": int(raw.byte_hash.nunique()),\n              "decoded_unique_before_quarantine": int(raw.image_id.nunique()),\n              "conflicting_exact_images_quarantined": len(bad), "images": len(clean),\n              "similarity_groups": int(clean.group_id.nunique()),\n              "class_counts": {str(k): int(v) for k, v in clean.label.value_counts().items()},\n              "invalid_files": invalid, "grouping_rule": "pHash distance <= 4 and Pearson r >= 0.98 at 32x32",\n              "group_unit": "image similarity; patient identity unknown",\n              "manifest_sha256": sha(clean.to_csv(index=False).encode())}\n    write_json(out / "dataset_audit.json", report)\n    print("DATA AUDIT", json.dumps(report), flush=True)\n    return clean, report\n\n\ndef locked_splits(df, n_folds=5, seed=42):\n    splits, records = [], []\n    outer = StratifiedGroupKFold(n_splits=n_folds, shuffle=True, random_state=seed)\n    for fold, (dev, test) in enumerate(outer.split(df, df.y, df.group_id)):\n        inner = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=seed + 1000 + fold)\n        tr, va = next(inner.split(df.iloc[dev], df.iloc[dev].y, df.iloc[dev].group_id))\n        parts = {"train": dev[tr], "validation": dev[va], "test": test}\n        sets = [set(df.iloc[v].group_id) for v in parts.values()]\n        assert not (sets[0] & sets[1] or sets[0] & sets[2] or sets[1] & sets[2])\n        assert all(df.iloc[ix].y.nunique() == 2 for ix in parts.values()), "Class missing from partition"\n        splits.append(parts)\n        for role, indices in parts.items():\n            for idx in indices:\n                records.append({"fold": fold, "role": role, "image_id": df.iloc[idx].image_id,\n                                "group_id": df.iloc[idx].group_id, "y": int(df.iloc[idx].y)})\n    return splits, pd.DataFrame(records)\n\n\ndef load_tensor(root, df, size):\n    images = []\n    for rel in df.relative_path:\n        with Image.open(Path(root) / rel) as im:\n            im = ImageOps.exif_transpose(im).convert("L")\n            im = ImageOps.pad(im, (size, size), method=Image.Resampling.BILINEAR, color=0)\n            a = np.asarray(im, dtype=np.float32)\n        mask = a > np.percentile(a, 10)\n        if mask.sum() < 100:\n            mask = np.ones_like(a, dtype=bool)\n        a = np.clip((a - a[mask].mean()) / max(float(a[mask].std()), 1e-6), -5, 5)\n        images.append(a)\n    return torch.from_numpy(np.stack(images)[:, None])\n\n\ndef augment(x, generator):\n    n = len(x)\n    rand = lambda *s: torch.rand(*s, generator=generator, device=x.device)\n    angle = (rand(n) * 2 - 1) * math.pi / 18\n    flip = torch.where(rand(n) < 0.5, -1., 1.)\n    t = torch.zeros(n, 2, 3, device=x.device)\n    t[:, 0, 0], t[:, 0, 1] = angle.cos() * flip, -angle.sin()\n    t[:, 1, 0], t[:, 1, 1] = angle.sin() * flip, angle.cos()\n    grid = F.affine_grid(t, x.size(), align_corners=False)\n    x = F.grid_sample(x, grid, mode="bilinear", padding_mode="border", align_corners=False)\n    return x * (0.9 + rand(n, 1, 1, 1) * 0.2) + (rand(n, 1, 1, 1) - 0.5) * 0.2\n\n\ndef model_for(name):\n    if name == "plain_cnn":\n        return Plain2DCNN(2)\n    if name == "narrow_resnet_gn":\n        return ResNet2D18(2)\n    return MSCANet2D(2, use_attention="no_attention" not in name,\n                    multiscale="no_multiscale" not in name)\n\n\ndef metrics(y, p):\n    pred = np.asarray(p) >= 0.5\n    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()\n    return {"macro_f1": float(f1_score(y, pred, average="macro", zero_division=0)),\n            "accuracy": float(accuracy_score(y, pred)),\n            "balanced_accuracy": float(balanced_accuracy_score(y, pred)),\n            "sensitivity": float(tp / (tp + fn)) if tp + fn else None,\n            "specificity": float(tn / (tn + fp)) if tn + fp else None,\n            "auroc": float(roc_auc_score(y, p)) if len(np.unique(y)) == 2 else None,\n            "average_precision": float(average_precision_score(y, p)) if np.sum(y) else None,\n            "brier": float(brier_score_loss(y, p)),\n            "log_loss": float(log_loss(y, np.column_stack([1 - p, p]), labels=[0, 1]))}\n\n\n@torch.no_grad()\ndef predict(model, x, indices, batch=64):\n    model.eval()\n    return np.concatenate([model(x[ix]).float().softmax(1)[:, 1].cpu().numpy()\n                           for ix in np.array_split(indices, max(1, math.ceil(len(indices) / batch)))])\n\n\ndef train_one(name, seed, fold, parts, x, y, df, args, split_hash):\n    run_dir = args.out / "runs" / f"{name}_s{seed}_f{fold}"\n    run_dir.mkdir(parents=True, exist_ok=True)\n    done = run_dir / "metrics.json"\n    run_config = {"model": name, "seed": seed, "fold": fold, "epochs": args.epochs,\n                  "patience": args.patience, "size": args.size, "batch": args.batch,\n                  "split_hash": split_hash, "protocol": "classification-v2"}\n    config_hash = sha(json.dumps(run_config, sort_keys=True).encode())\n    if done.exists():\n        prior = json.loads(done.read_text())\n        if prior.get("config_hash") != config_hash:\n            raise RuntimeError("Resume configuration differs from completed run")\n        return prior, pd.read_csv(run_dir / "predictions.csv")\n    random.seed(seed + fold * 1000); np.random.seed(seed + fold * 1000)\n    torch.manual_seed(seed + fold * 1000); torch.cuda.manual_seed_all(seed + fold * 1000)\n    model = model_for(name).to(x.device)\n    opt = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)\n    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=args.epochs)\n    counts = np.bincount(y[parts["train"]].cpu().numpy(), minlength=2)\n    weights = torch.tensor(counts.sum() / (2 * counts), dtype=torch.float32, device=x.device)\n    criterion = nn.CrossEntropyLoss(weight=weights, label_smoothing=0.05)\n    amp = x.device.type == "cuda"\n    scaler = torch.amp.GradScaler("cuda", enabled=amp)\n    best_score, best_loss, stale, history = -1., float("inf"), 0, []\n    best_state, best_epoch = None, None\n    start = time.monotonic()\n    if amp:\n        torch.cuda.reset_peak_memory_stats()\n    for epoch in range(args.epochs):\n        model.train()\n        rng = np.random.default_rng([seed, fold, epoch])\n        indices = rng.permutation(parts["train"])\n        aug_rng = torch.Generator(device=x.device).manual_seed(seed * 100000 + fold * 1000 + epoch)\n        losses = []\n        for begin in range(0, len(indices), args.batch):\n            ix = indices[begin:begin + args.batch]\n            bx = x[ix]\n            if name != "no_augmentation":\n                bx = augment(bx, aug_rng)\n            opt.zero_grad(set_to_none=True)\n            with torch.autocast(device_type=x.device.type, enabled=amp):\n                loss = criterion(model(bx), y[ix])\n            if not torch.isfinite(loss):\n                raise FloatingPointError("Non-finite classification loss")\n            scaler.scale(loss).backward(); scaler.unscale_(opt)\n            nn.utils.clip_grad_norm_(model.parameters(), 1.0)\n            scaler.step(opt); scaler.update()\n            losses.append(float(loss.detach()))\n        sched.step()\n        p = predict(model, x, parts["validation"])\n        m = metrics(y[parts["validation"]].cpu().numpy(), p)\n        score, val_loss = m["macro_f1"], m["log_loss"]\n        history.append({"epoch": epoch + 1, "train_loss": float(np.mean(losses)),\n                        "validation_macro_f1": score, "validation_log_loss": val_loss})\n        if score > best_score + 1e-9 or (abs(score - best_score) < 1e-9 and val_loss < best_loss - 1e-6):\n            best_score, best_loss, stale, best_epoch = score, val_loss, 0, epoch + 1\n            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}\n        else:\n            stale += 1\n        if stale >= args.patience:\n            break\n    assert best_state is not None\n    model.load_state_dict(best_state)\n    pred = predict(model, x, parts["test"])\n    frame = df.iloc[parts["test"]][["image_id", "group_id", "y"]].copy()\n    frame["model"], frame["seed"], frame["fold"], frame["probability_yes"] = name, seed, fold, pred\n    frame.to_csv(run_dir / "predictions.csv", index=False)\n    pd.DataFrame(history).to_csv(run_dir / "history.csv", index=False)\n    ckpts = args.out / "checkpoints"; ckpts.mkdir(exist_ok=True)\n    torch.save({"state_dict": best_state, "config": run_config}, ckpts / f"{name}_s{seed}_f{fold}.pt")\n    result = {**run_config, **metrics(frame.y.values, pred), "config_hash": config_hash,\n              "best_epoch": best_epoch, "epochs_run": len(history),\n              "inner_validation_macro_f1": best_score,\n              "parameters": sum(p.numel() for p in model.parameters()),\n              "wall_seconds": time.monotonic() - start,\n              "peak_gpu_gb": torch.cuda.max_memory_allocated() / 2**30 if amp else 0.}\n    write_json(done, result)\n    print(json.dumps(result), flush=True)\n    del model\n    if amp:\n        torch.cuda.empty_cache()\n    return result, frame\n\n\ndef summarize(predictions, out, bootstrap=2000):\n    """Descriptive group bootstrap of mean seed-wise OOF F1, never seed pseudoreplication."""\n    rows = []\n    for (name, seed), f in predictions.groupby(["model", "seed"]):\n        rows.append({"model": name, "seed": int(seed), **metrics(f.y.values, f.probability_yes.values)})\n    per_seed = pd.DataFrame(rows)\n    per_seed.to_csv(out / "per_seed_oof_metrics.csv", index=False)\n    summary = per_seed.groupby("model").agg({k: ["mean", "std"] for k in metrics(np.array([0, 1]), np.array([0.2, 0.8]))})\n    summary.columns = ["_".join(c) for c in summary.columns]\n    summary.to_csv(out / "model_summary.csv")\n    # Identical image ordering makes paired resampling explicit and checkable.\n    ids = sorted(predictions.image_id.unique())\n    meta = predictions.drop_duplicates("image_id").set_index("image_id").loc[ids]\n    groups = sorted(meta.group_id.unique())\n    members = [np.where(meta.group_id.values == g)[0] for g in groups]\n    model_names = sorted(predictions.model.unique())\n    arrays = {}\n    for name in model_names:\n        matrix = predictions[predictions.model == name].pivot(index="image_id", columns="seed", values="probability_yes").reindex(ids)\n        if matrix.isna().any().any():\n            raise ValueError("Incomplete OOF matrix")\n        arrays[name] = matrix.values >= 0.5\n    y = meta.y.values\n\n    def fast_f1(ix, name):\n        pr, truth = arrays[name][ix], y[ix, None].astype(bool)\n        tp = (pr & truth).sum(0); tn = (~pr & ~truth).sum(0)\n        fp = (pr & ~truth).sum(0); fn = (~pr & truth).sum(0)\n        return float(np.mean((2*tp / np.maximum(2*tp+fp+fn, 1) + 2*tn / np.maximum(2*tn+fp+fn, 1)) / 2))\n    rng = np.random.default_rng(20260909)\n    draws = {n: [] for n in model_names}\n    for _ in range(bootstrap):\n        ix = np.concatenate([members[g] for g in rng.integers(len(groups), size=len(groups))])\n        for name in model_names:\n            draws[name].append(fast_f1(ix, name))\n    cis, contrasts = [], []\n    for name in model_names:\n        lo, hi = np.quantile(draws[name], [0.025, 0.975])\n        cis.append({"model": name, "mean_seed_oof_macro_f1": fast_f1(np.arange(len(y)), name),\n                    "ci_low": float(lo), "ci_high": float(hi), "bootstrap_groups": len(groups)})\n        if name != "full":\n            d = np.asarray(draws["full"]) - np.asarray(draws[name])\n            lo, hi = np.quantile(d, [0.025, 0.975])\n            contrasts.append({"comparison": f"full - {name}",\n                              "macro_f1_delta": fast_f1(np.arange(len(y)), "full") - fast_f1(np.arange(len(y)), name),\n                              "ci_low": float(lo), "ci_high": float(hi)})\n    pd.DataFrame(cis).to_csv(out / "descriptive_group_bootstrap.csv", index=False)\n    pd.DataFrame(contrasts).to_csv(out / "paired_ablation_deltas.csv", index=False)\n    write_json(out / "inference_limitations.json", {"scope": "image-level exploratory benchmark",\n        "bootstrap": "paired similarity groups; seeds averaged within each draw",\n        "limitation": "unknown patients and dependent cross-validation fits; intervals are descriptive, not clinical or confirmatory significance"})\n    import matplotlib\n    matplotlib.use("Agg")\n    import matplotlib.pyplot as plt\n    c = pd.DataFrame(cis).sort_values("mean_seed_oof_macro_f1")\n    fig, ax = plt.subplots(figsize=(9, 4.5))\n    ax.errorbar(c.mean_seed_oof_macro_f1, c.model,\n                xerr=[c.mean_seed_oof_macro_f1 - c.ci_low, c.ci_high - c.mean_seed_oof_macro_f1], fmt="o", capsize=3)\n    ax.set_xlabel("Mean seed-wise out-of-fold macro F1 (descriptive 95% group bootstrap)")\n    ax.set_xlim(0, 1); fig.tight_layout()\n    fig.savefig(out / "classification_comparison.png", dpi=250)\n    fig.savefig(out / "classification_comparison.pdf")\n    plt.close(fig)\n    print(summary.round(4).to_string(), flush=True)\n\n\ndef main(argv=None):\n    ap = argparse.ArgumentParser()\n    ap.add_argument("--root", type=Path, required=True)\n    ap.add_argument("--out", type=Path, default=Path("classification_results"))\n    ap.add_argument("--epochs", type=int, default=60)\n    ap.add_argument("--patience", type=int, default=12)\n    ap.add_argument("--size", type=int, default=128)\n    ap.add_argument("--batch", type=int, default=32)\n    ap.add_argument("--seeds", type=int, nargs="+", default=[42, 43, 44])\n    ap.add_argument("--models", nargs="+", default=VARIANTS, choices=VARIANTS)\n    ap.add_argument("--audit-only", action="store_true")\n    args = ap.parse_args(argv)\n    args.out.mkdir(parents=True, exist_ok=True)\n    torch.set_num_threads(min(4, os.cpu_count() or 1))\n    torch.backends.cudnn.benchmark = False; torch.backends.cudnn.deterministic = True\n    df, report = audit_images(args.root, args.out / "audit")\n    parts, split_df = locked_splits(df)\n    split_df.to_csv(args.out / "split_manifest.csv", index=False)\n    split_hash = sha(split_df.to_csv(index=False).encode())\n    write_json(args.out / "protocol_lock.json", {"split_sha256": split_hash,\n        "dataset_sha256": report["manifest_sha256"], "models": args.models, "seeds": args.seeds,\n        "epochs": args.epochs, "patience": args.patience, "size": args.size,\n        "primary_metric": "macro_f1", "unit": "image similarity group; not patient",\n        "preprocessing": "aspect-preserving letterbox, per-image foreground z-score"})\n    if args.audit_only:\n        return\n    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")\n    write_json(args.out / "environment.json", {"python": platform.python_version(),\n        "torch": torch.__version__, "numpy": np.__version__, "pandas": pd.__version__,\n        "device": str(device), "gpu": torch.cuda.get_device_name() if device.type == "cuda" else None})\n    x = load_tensor(args.root, df, args.size).to(device)\n    y = torch.tensor(df.y.values, dtype=torch.long, device=device)\n    results, frames = [], []\n    for seed in args.seeds:\n        for fold, partition in enumerate(parts):\n            for name in args.models:\n                r, f = train_one(name, seed, fold, partition, x, y, df, args, split_hash)\n                results.append(r); frames.append(f)\n                pd.DataFrame(results).to_csv(args.out / "per_run_metrics.csv", index=False)\n    predictions = pd.concat(frames, ignore_index=True)\n    predictions.to_csv(args.out / "oof_predictions.csv", index=False)\n    summarize(predictions, args.out)\n    write_json(args.out / "completion.json", {"completed": True, "runs": len(results),\n        "expected_runs": len(args.seeds) * 5 * len(args.models),\n        "scope": "repeated-seed image benchmark; conference/clinical validation not established"})\n\n\nif __name__ == "__main__":\n    main()\n'}
SOURCE_SHA256 = '77cc130fba112f2737a5f3645c81c88ce2198a24d3002f8e51ba5284c4b36add'
assert hashlib.sha256(json.dumps(SOURCES, sort_keys=True).encode()).hexdigest() == SOURCE_SHA256
for rel, text in SOURCES.items():
    path = SOURCE_ROOT / rel
    assert path.resolve().is_relative_to(SOURCE_ROOT.resolve())
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(text)
sys.path.insert(0, str(SOURCE_ROOT))
sys.path.insert(0, str(SOURCE_ROOT / 'segmentation' / 'src'))
print('Source SHA256:', SOURCE_SHA256)


## Run the frozen protocol

In [ ]:
from classification.study import main
roots = [p for p in Path('/kaggle/input').rglob('brain-mri-images-for-brain-tumor-detection') if p.is_dir()]
if len(roots) != 1:
    raise RuntimeError(f'Expected one attached classification dataset, found {roots}')
main(['--root', str(roots[0]), '--out', str(WORK / 'classification_results'),
      '--epochs', '60', '--patience', '12', '--batch', '32', '--seeds', '42', '43', '44'])
